<a href="https://colab.research.google.com/github/zranadegirmenci/sql-veri-analizi-projeleri/blob/main/sql_ile_musteri_ve_satis_analitigi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Bellekte ilişkisel SQLite veritabanı oluşturma
baglanti = sqlite3.connect('sirket_analitigi.db')
cursor = baglanti.cursor()

# 1. Müşteriler Tablosu
cursor.execute('''
CREATE TABLE IF NOT EXISTS musteriler (
    musteri_id INTEGER PRIMARY KEY,
    ad_soyad TEXT,
    sehir TEXT,
    kayit_tarihi DATE
)
''')

# 2. Siparişler Tablosu (Foreign Key ile bağlı)
cursor.execute('''
CREATE TABLE IF NOT EXISTS siparisler (
    siparis_id INTEGER PRIMARY KEY,
    musteri_id INTEGER,
    kategori TEXT,
    tutar REAL,
    siparis_tarihi DATE,
    FOREIGN KEY (musteri_id) REFERENCES musteriler (musteri_id)
)
''')

# Örnek Gerçekçi Verilerin Eklenmesi
musteri_verileri = [
    (1, 'Ahmet Yılmaz', 'İstanbul', '2023-01-15'),
    (2, 'Ayşe Kaya', 'Ankara', '2023-02-10'),
    (3, 'Mehmet Demir', 'İzmir', '2023-03-05'),
    (4, 'Fatma Çelik', 'Bursa', '2023-04-12'),
    (5, 'Caner Öz', 'Antalya', '2023-05-20'),
    (6, 'Zeynep Şahin', 'İstanbul', '2023-06-01')
]

siparis_verileri = [
    (101, 1, 'Elektronik', 4500.0, '2023-06-10'),
    (102, 1, 'Aksesuar', 350.0, '2023-07-15'),
    (103, 2, 'Giyim', 1200.0, '2023-06-20'),
    (104, 3, 'Elektronik', 8900.0, '2023-07-02'),
    (105, 3, 'Kitap', 250.0, '2023-08-11'),
    (106, 4, 'Ev & Yaşam', 2300.0, '2023-08-15'),
    (107, 5, 'Elektronik', 15000.0, '2023-09-01'),
    (108, 1, 'Giyim', 800.0, '2023-09-05'),
    (109, 2, 'Aksesuar', 450.0, '2023-09-12')
]

cursor.executemany('INSERT OR IGNORE INTO musteriler VALUES (?, ?, ?, ?)', musteri_verileri)
cursor.executemany('INSERT OR IGNORE INTO siparisler VALUES (?, ?, ?, ?, ?)', siparis_verileri)
baglanti.commit()

print("Veritabanı, ilişkisel tablolar ve veriler başarıyla yüklendi!")

Veritabanı, ilişkisel tablolar ve veriler başarıyla yüklendi!


# 1. Müşteri Bazlı Toplam Harcama ve Sipariş Hacmi Analizi (INNER JOIN & GROUP BY)
İki ilişkisel tablo (müşteriler ve siparişler) müşteri ID üzerinden birleştirilerek her müşterinin toplam işlem hacmi, toplam harcaması ve ortalama sepet büyüklüğü hesaplanmıştır.

In [12]:
sorgu_1 = '''
SELECT
    m.musteri_id,
    m.ad_soyad,
    m.sehir,
    COUNT(s.siparis_id) AS toplam_siparis_sayisi,
    ROUND(SUM(s.tutar), 2) AS toplam_harcama,
    ROUND(AVG(s.tutar), 2) AS ortalama_sepet_tutari
FROM musteriler m
INNER JOIN siparisler s ON m.musteri_id = s.musteri_id
GROUP BY m.musteri_id, m.ad_soyad, m.sehir
ORDER BY toplam_harcama DESC;
'''

df_musteri_ozeti = pd.read_sql_query(sorgu_1, baglanti)
print("--- Müşteri Harcama Performansı Tablosu ---")
df_musteri_ozeti

--- Müşteri Harcama Performansı Tablosu ---


,musteri_id,ad_soyad,sehir,toplam_siparis_sayisi,toplam_harcama,ortalama_sepet_tutari
0,5,Caner Öz,Antalya,1,15000.0,15000.00
1,3,Mehmet Demir,İzmir,2,9150.0,4575.00
2,1,Ahmet Yılmaz,İstanbul,3,5650.0,1883.33
3,4,Fatma Çelik,Bursa,1,2300.0,2300.00
4,2,Ayşe Kaya,Ankara,2,1650.0,825.00


# 2. Yüksek Değerli (VIP) Müşterilerin Tespiti (HAVING Koşulu)
Agregasyon sonrası toplam harcaması 2000 TL'nin üzerinde olan veya 2'den fazla sipariş vermiş yüksek değerli müşteri segmenti filtrelenmiştir.

In [13]:
sorgu_2 = '''
SELECT
    m.ad_soyad,
    m.sehir,
    COUNT(s.siparis_id) AS siparis_adedi,
    ROUND(SUM(s.tutar), 2) AS toplam_ciro
FROM musteriler m
INNER JOIN siparisler s ON m.musteri_id = s.musteri_id
GROUP BY m.musteri_id, m.ad_soyad, m.sehir
HAVING SUM(s.tutar) > 2000 OR COUNT(s.siparis_id) >= 2
ORDER BY toplam_ciro DESC;
'''

df_vip = pd.read_sql_query(sorgu_2, baglanti)
print("--- VIP Müşteri Segmenti ---")
df_vip

--- VIP Müşteri Segmenti ---


,ad_soyad,sehir,siparis_adedi,toplam_ciro
0,Caner Öz,Antalya,1,15000.0
1,Mehmet Demir,İzmir,2,9150.0
2,Ahmet Yılmaz,İstanbul,3,5650.0
3,Fatma Çelik,Bursa,1,2300.0
4,Ayşe Kaya,Ankara,2,1650.0


# 3. Kategori Bazlı En Yüksek Siparişlerin Sıralanması (Pencere Fonksiyonu / DENSE_RANK)
Her ürün kategorisinde yapılan siparişler kendi içinde tutara göre sıralanmış ve her kategorinin en yüksek gelir getiren işlemleri listelenmiştir.

In [14]:
sorgu_3 = '''
SELECT
    kategori,
    siparis_id,
    musteri_id,
    tutar,
    DENSE_RANK() OVER(PARTITION BY kategori ORDER BY tutar DESC) AS kategori_ici_sira
FROM siparisler;
'''

df_siralama = pd.read_sql_query(sorgu_3, baglanti)
print("--- Kategori İçi Harcama Sıralamaları (Window Function) ---")
df_siralama

--- Kategori İçi Harcama Sıralamaları (Window Function) ---


,kategori,siparis_id,musteri_id,tutar,kategori_ici_sira
0,Aksesuar,109,2,450.0,1
1,Aksesuar,102,1,350.0,2
2,Elektronik,107,5,15000.0,1
3,Elektronik,104,3,8900.0,2
4,Elektronik,101,1,4500.0,3
5,Ev & Yaşam,106,4,2300.0,1
6,Giyim,103,2,1200.0,1
7,Giyim,108,1,800.0,2
8,Kitap,105,3,250.0,1


In [15]:
# Veritabanı bağlantısını güvenli kapatma
baglanti.close()
print("Veritabanı bağlantısı başarıyla kapatıldı.")

Veritabanı bağlantısı başarıyla kapatıldı.


# 4. SQL Tabanlı İş Zekası ve Gelir Dağılımı Özeti (Executive Insights)

İlişkisel veritabanı üzerinde yürütülen sorgulamalar sonucunda operasyonel ve ticari karar süreçleri için aşağıdaki bulgular elde edilmiştir:

* **Müşteri Yoğunlaşması ve Gelir Riski:** Toplam cironun önemli bir kısmı az sayıda yüksek hacimli (VIP) müşteriden gelmektedir. Bu müşterilere özel sadakat programları ve kişiselleştirilmiş hesap yöneticisi desteği sağlanmalıdır.
* **Kategori Performansı ve Sepet Büyüklüğü:** Elektronik kategorisi en yüksek tekil sepet tutarlarına sahipken; Giyim ve Aksesuar kategorileri daha sık fakat daha düşük tutarlı siparişlerle sepet frekansını desteklemektedir. Çapraz satış (cross-selling) kurguları Elektronik ürün satın alanlara Aksesuar önerisi üzerine optimize edilmelidir.
* **Teknik Altyapı Notu:** Veriler ilişkisel veritabanı kurallarına uygun olarak normalize edilmiş, `INNER JOIN`, `HAVING` ve pencere fonksiyonları (`DENSE_RANK`) ile çok boyutlu iş soruları SQL seviyesinde optimize edilerek yanıtlanmıştır.